# 单井 gain 曲线普通克里金模型 QC

名义上对接 `wavelet_amplitude_gain_curve_depth@20260428.ipynb` 的输出；当前实验只使用 `2-ANP-2A-RJS` 的 `gain_depth_curve` 作为唯一井控，构建深度域层位约束 gain 模型。

建模后，在每口井位置抽取模型 gain 曲线，映射回各井合成记录 QC 的时间采样，生成施加模型 gain 后的 synthetic QC。


In [ ]:
import json
import sys
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_interpretation_petrel
from cup.seismic.modeling import WellControl, build_layer_constrained_model
from cup.seismic.survey import open_survey
from cup.seismic.target_layer import TargetLayer
from wtie.processing import grid

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)


## 1) 配置输入输出


In [ ]:
data_root = repo_root / "data"

batch_output_dir = data_root / "output_wavelet_batch_synthetic_depth_20260428"
batch_metrics_file = batch_output_dir / "wavelet_batch_metrics.csv"

gain_curve_output_dir = data_root / "output_wavelet_amplitude_gain_curve_depth_20260428"
gain_curve_metrics_file = gain_curve_output_dir / "wavelet_amplitude_gain_curve_metrics.csv"
control_well_name = "2-ANP-2A-RJS"

output_dir = data_root / "output_wavelet_amplitude_gain_model_20260428"
well_qc_dir = output_dir / "well_qc"
figure_dir = output_dir / "figures"
model_output_file = output_dir / "wavelet_amplitude_gain_model.npz"
well_qc_metrics_file = output_dir / "wavelet_amplitude_gain_model_well_qc_metrics.csv"

train_config_file = repo_root / "experiments" / "ginn_depth" / "train.yaml"
seismic_file = data_root / "raw" / "mero_84_coord_extend"
segy_options = {
    "iline": 5,
    "xline": 21,
    "istep": 1,
    "xstep": 4,
}

horizon_files = {
    "base_of_salt": data_root / "interpre_depth" / "base_of_salt_extend",
    "base_of_bve": data_root / "interpre_depth" / "base_of_bve_extend",
    "base_of_itp": data_root / "interpre_depth" / "base_of_itp_extend",
}

model_params = {
    "boundary_extension_samples": 50,
    "n_slices": 20,
    "variogram": "spherical",
    "exact": True,
    "nugget": 0.0,
    "post_slice_smoothing": True,
}

gain_floor = 1e-6

required_paths = [batch_metrics_file, gain_curve_metrics_file, train_config_file, seismic_file, *horizon_files.values()]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

output_dir.mkdir(parents=True, exist_ok=True)
well_qc_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

batch_metrics_df = pd.read_csv(batch_metrics_file)
batch_ok_df = batch_metrics_df.loc[batch_metrics_df["status"] == "ok"].copy()
if batch_ok_df.empty:
    raise ValueError("No successful wells found in batch metrics.")

gain_curve_metrics_df = pd.read_csv(gain_curve_metrics_file)
control_gain_row = gain_curve_metrics_df.loc[
    (gain_curve_metrics_df["well_name"] == control_well_name) & (gain_curve_metrics_df["status"] == "ok")
]
if control_gain_row.empty:
    raise ValueError(f"No successful gain curve found for control well {control_well_name!r}.")
control_gain_row = control_gain_row.iloc[0]
control_gain_curve_path = Path(control_gain_row["gain_depth_curve_path"])
if not control_gain_curve_path.is_absolute():
    control_gain_curve_path = repo_root / control_gain_curve_path
if not control_gain_curve_path.exists():
    raise FileNotFoundError(control_gain_curve_path)

with train_config_file.open("r", encoding="utf-8") as file:
    train_config = yaml.safe_load(file)

target_layer_config = {
    "min_thickness": train_config.get("target_layer_min_thickness"),
    "nearest_distance_limit": train_config.get("target_layer_nearest_distance_limit"),
    "outlier_threshold": train_config.get("target_layer_outlier_threshold"),
    "outlier_min_neighbor_count": train_config.get("target_layer_outlier_min_neighbor_count"),
}

print("Control gain curve:", control_gain_curve_path)
print("Batch wells:", batch_ok_df["well_name"].tolist())
print("Output dir:", output_dir)


## 2) 工区几何与目标层


In [ ]:
survey = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options=segy_options,
)
geometry_depth = survey.query_geometry(domain="depth")

print(json.dumps(geometry_depth, indent=2, ensure_ascii=False))
assert geometry_depth["sample_domain"] == "depth"
assert geometry_depth["sample_unit"] == "m"

def build_target_layer(geometry: dict) -> TargetLayer:
    raw_horizons = {
        horizon_name: import_interpretation_petrel(horizon_file)
        for horizon_name, horizon_file in horizon_files.items()
    }
    return TargetLayer(
        raw_horizon_dfs=raw_horizons,
        geometry=geometry,
        horizon_names=list(horizon_files.keys()),
        min_thickness=target_layer_config["min_thickness"],
        nearest_distance_limit=target_layer_config["nearest_distance_limit"],
        outlier_threshold=target_layer_config["outlier_threshold"],
        outlier_min_neighbor_count=target_layer_config["outlier_min_neighbor_count"],
    )


target_layer = build_target_layer(geometry_depth)
print("horizons:", target_layer.horizon_names)
print("zones:", target_layer.iter_zones())
display(target_layer.qc_summary_df)


## 3) 构建单井 gain 井控


In [ ]:
def safe_name(well_name: str) -> str:
    return well_name.replace("/", "_").replace("\\", "_").replace(" ", "_")


def resolve_artifact_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def compute_trace_metrics(reference: np.ndarray, prediction: np.ndarray, mask: np.ndarray) -> dict:
    reference = np.asarray(reference, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    mask = np.asarray(mask, dtype=bool) & np.isfinite(reference) & np.isfinite(prediction)
    n = int(mask.sum())
    if n < 5:
        return {"corr": np.nan, "nmae": np.nan, "n_samples": n}
    ref = reference[mask]
    pred = prediction[mask]
    corr = np.nan if np.std(ref) <= 0.0 or np.std(pred) <= 0.0 else float(np.corrcoef(ref, pred)[0, 1])
    denom = float(np.sum(np.abs(ref)))
    nmae = np.nan if denom <= 0.0 else float(np.sum(np.abs(ref - pred)) / denom)
    return {"corr": corr, "nmae": nmae, "n_samples": n}


def twt_to_tvdss(twt_s: np.ndarray, depth_shift_df: pd.DataFrame) -> np.ndarray:
    depth_twt = depth_shift_df["twt_s"].to_numpy(dtype=float)
    depth_z = depth_shift_df["tvdss_m"].to_numpy(dtype=float)
    finite = np.isfinite(depth_twt) & np.isfinite(depth_z)
    depth_twt = depth_twt[finite]
    depth_z = depth_z[finite]
    if depth_twt.size < 2:
        raise ValueError("Depth shift curve has too few finite samples.")
    order = np.argsort(depth_twt)
    depth_twt = depth_twt[order]
    depth_z = depth_z[order]
    unique_twt, unique_idx = np.unique(depth_twt, return_index=True)
    unique_z = depth_z[unique_idx]
    return np.interp(twt_s, unique_twt, unique_z, left=unique_z[0], right=unique_z[-1])


def nearest_model_trace(result, inline: float, xline: float) -> tuple[np.ndarray, float, float]:
    il_idx = int(np.argmin(np.abs(result.ilines - float(inline))))
    xl_idx = int(np.argmin(np.abs(result.xlines - float(xline))))
    return np.asarray(result.volume[il_idx, xl_idx, :], dtype=float), float(result.ilines[il_idx]), float(result.xlines[xl_idx])


control_batch_row = batch_ok_df.loc[batch_ok_df["well_name"] == control_well_name]
if control_batch_row.empty:
    raise ValueError(f"Control well {control_well_name!r} not found in batch metrics.")
control_batch_row = control_batch_row.iloc[0]
control_inline = float(control_batch_row["inline_float"])
control_xline = float(control_batch_row["xline_float"])
control_horizons = target_layer.get_interpretation_values_at_location(control_inline, control_xline)

gain_depth_df = pd.read_csv(control_gain_curve_path)
required_gain_columns = {"tvdss_m", "gain_smooth"}
missing = required_gain_columns - set(gain_depth_df.columns)
if missing:
    raise ValueError(f"Control gain curve missing columns: {sorted(missing)}")

gain_z = gain_depth_df["tvdss_m"].to_numpy(dtype=float)
gain_values = gain_depth_df["gain_smooth"].to_numpy(dtype=float)
finite = np.isfinite(gain_z) & np.isfinite(gain_values) & (gain_values > 0.0)
gain_z = gain_z[finite]
gain_values = gain_values[finite]
order = np.argsort(gain_z)
gain_z = gain_z[order]
gain_values = gain_values[order]
unique_z, unique_idx = np.unique(gain_z, return_index=True)
unique_gain = gain_values[unique_idx]
if unique_z.size < 2:
    raise ValueError("Control gain curve has too few unique depth samples.")

control_gain_log = grid.Log(
    unique_gain.astype(np.float64),
    unique_z.astype(np.float64),
    "tvdss",
    name="WaveletAmplitudeGainFromANP",
    unit="scale",
    allow_nan=False,
)
control = WellControl(
    well_name=control_well_name,
    property_name="WaveletAmplitudeGainFromANP",
    property_log=control_gain_log,
    inline=control_inline,
    xline=control_xline,
    horizon_values={name: float(value) for name, value in control_horizons.items()},
    metadata={
        "gain_curve_path": str(control_gain_curve_path),
        "source": "wavelet_amplitude_gain_curve_depth@20260428",
    },
)

print("Control:", control.well_name, control.inline, control.xline)
print("Gain depth range:", float(unique_z[0]), "to", float(unique_z[-1]), "n=", unique_z.size)


## 4) 普通克里金建模


In [ ]:
gain_result = build_layer_constrained_model(
    target_layer=target_layer,
    well_controls=[control],
    **model_params,
)
finite_volume = np.isfinite(gain_result.volume)
non_positive_count = int(np.count_nonzero(finite_volume & (gain_result.volume <= gain_floor)))
if non_positive_count > 0:
    gain_result.volume = np.where(
        finite_volume,
        np.maximum(gain_result.volume, gain_floor),
        gain_result.volume,
    ).astype(np.float32)
gain_result.metadata["control_well_name"] = control_well_name
gain_result.metadata["gain_floor"] = float(gain_floor)
gain_result.metadata["non_positive_values_clipped"] = non_positive_count

np.savez_compressed(
    model_output_file,
    volume=gain_result.volume.astype(np.float32),
    variance_volume=gain_result.variance_volume.astype(np.float32),
    ilines=gain_result.ilines,
    xlines=gain_result.xlines,
    samples=gain_result.samples,
    geometry_json=json.dumps(gain_result.geometry, ensure_ascii=False),
    metadata_json=json.dumps(gain_result.metadata, ensure_ascii=False),
    coverage_stats_json=json.dumps(gain_result.coverage_stats, ensure_ascii=False),
)
print("Saved model:", model_output_file)
print("volume shape:", gain_result.volume.shape)
print("non-positive values clipped:", non_positive_count)
print("coverage modes:", gain_result.coverage_stats["zones"])


## 5) 模型切片快速 QC


In [ ]:
finite_values = gain_result.volume[np.isfinite(gain_result.volume)]
if finite_values.size == 0:
    raise ValueError("Gain model has no finite samples.")
vmin, vmax = np.nanpercentile(finite_values, [5, 95])

i_il = len(gain_result.ilines) // 2
i_xl = len(gain_result.xlines) // 2
i_z = len(gain_result.samples) // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
im0 = axes[0].imshow(
    gain_result.volume[i_il, :, :].T,
    aspect="auto",
    origin="upper",
    extent=[gain_result.xlines[0], gain_result.xlines[-1], gain_result.samples[-1], gain_result.samples[0]],
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
axes[0].set_title(f"Gain inline @ {gain_result.ilines[i_il]:.0f}")
axes[0].set_xlabel("Xline")
axes[0].set_ylabel("TVDSS depth (m)")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(
    gain_result.volume[:, i_xl, :].T,
    aspect="auto",
    origin="upper",
    extent=[gain_result.ilines[0], gain_result.ilines[-1], gain_result.samples[-1], gain_result.samples[0]],
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
axes[1].set_title(f"Gain xline @ {gain_result.xlines[i_xl]:.0f}")
axes[1].set_xlabel("Inline")
axes[1].set_ylabel("TVDSS depth (m)")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

im2 = axes[2].imshow(
    gain_result.volume[:, :, i_z].T,
    aspect="auto",
    origin="lower",
    extent=[gain_result.ilines[0], gain_result.ilines[-1], gain_result.xlines[0], gain_result.xlines[-1]],
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
axes[2].scatter([control_inline], [control_xline], c="red", s=35, label=control_well_name)
axes[2].set_title(f"Gain depth slice @ {gain_result.samples[i_z]:.1f} m")
axes[2].set_xlabel("Inline")
axes[2].set_ylabel("Xline")
axes[2].legend(loc="best")
fig.colorbar(im2, ax=axes[2], shrink=0.85)

model_fig = figure_dir / "qc_01_gain_model_from_anp_slices.png"
fig.savefig(model_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", model_fig)


## 6) 每口井抽取模型 gain 并重跑合成记录 QC


In [ ]:
well_qc_rows = []
for _, row in batch_ok_df.iterrows():
    well_name = str(row["well_name"])
    name = safe_name(well_name)
    print(f"\n=== {well_name} ===")
    try:
        synthetic_qc_path = resolve_artifact_path(row["synthetic_qc_path"])
        depth_shift_curve_path = resolve_artifact_path(row["depth_shift_curve_path"])
        qc_df = pd.read_csv(synthetic_qc_path)
        depth_shift_df = pd.read_csv(depth_shift_curve_path)

        twt_s = qc_df["twt_s"].to_numpy(dtype=float)
        seismic_norm = qc_df["seismic_norm"].to_numpy(dtype=float)
        synthetic_scaled = qc_df["synthetic_scaled"].to_numpy(dtype=float)
        eval_mask = qc_df["eval_mask"].astype(bool).to_numpy()
        scale = float(row["scale"])
        synthetic_raw = synthetic_scaled / scale
        tvdss_m = twt_to_tvdss(twt_s, depth_shift_df)

        model_trace, nearest_il, nearest_xl = nearest_model_trace(gain_result, float(row["inline_float"]), float(row["xline_float"]))
        model_gain = np.interp(tvdss_m, gain_result.samples, model_trace, left=np.nan, right=np.nan)
        synthetic_model_gain = synthetic_raw * model_gain

        constant_metrics = compute_trace_metrics(seismic_norm, synthetic_scaled, eval_mask)
        model_metrics = compute_trace_metrics(seismic_norm, synthetic_model_gain, eval_mask)
        gain_eval = model_gain[eval_mask & np.isfinite(model_gain)]

        qc_out = pd.DataFrame(
            {
                "twt_s": twt_s,
                "tvdss_m": tvdss_m,
                "seismic_norm": seismic_norm,
                "synthetic_raw": synthetic_raw,
                "synthetic_scaled": synthetic_scaled,
                "model_gain": model_gain,
                "synthetic_model_gain": synthetic_model_gain,
                "eval_mask": eval_mask,
            }
        )
        qc_path = well_qc_dir / f"model_gain_qc_{name}.csv"
        qc_out.to_csv(qc_path, index=False)

        fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
        t_ms = twt_s * 1000.0
        axes[0].plot(model_gain, t_ms, lw=1.1, color="tab:blue")
        axes[0].invert_yaxis()
        axes[0].set_xlabel("Model gain")
        axes[0].set_ylabel("Relative TWT (ms)")
        axes[0].set_title("gain(t)")
        axes[0].grid(True, alpha=0.25)

        axes[1].plot(model_gain, tvdss_m, lw=1.1, color="tab:green")
        axes[1].invert_yaxis()
        axes[1].set_xlabel("Model gain")
        axes[1].set_ylabel("TVDSS (m)")
        axes[1].set_title("gain(z)")
        axes[1].grid(True, alpha=0.25)

        axes[2].plot(seismic_norm, t_ms, lw=0.9, color="black", label="seismic")
        axes[2].plot(synthetic_scaled, t_ms, lw=0.9, color="tab:red", alpha=0.75, label="constant scale")
        axes[2].plot(synthetic_model_gain, t_ms, lw=0.9, color="tab:blue", alpha=0.85, label="model gain")
        axes[2].invert_yaxis()
        axes[2].set_xlabel("Normalized amplitude")
        axes[2].set_title("synthetic QC")
        axes[2].legend(loc="best", fontsize=8)
        axes[2].grid(True, alpha=0.25)

        fig.suptitle(
            f"{well_name}: corr {constant_metrics['corr']:.3f}->{model_metrics['corr']:.3f}, "
            f"nmae {constant_metrics['nmae']:.3f}->{model_metrics['nmae']:.3f}",
            y=1.02,
        )
        fig.tight_layout()
        fig_path = figure_dir / f"qc_{name}_model_gain_synthetic.png"
        fig.savefig(fig_path, dpi=180, bbox_inches="tight")
        plt.close(fig)

        metric_row = {
            "well_name": well_name,
            "status": "ok",
            "error": "",
            "scale": scale,
            "source_corr": float(row["corr"]),
            "source_nmae": float(row["nmae"]),
            "constant_corr": constant_metrics["corr"],
            "constant_nmae": constant_metrics["nmae"],
            "model_gain_corr": model_metrics["corr"],
            "model_gain_nmae": model_metrics["nmae"],
            "n_eval_samples": int(model_metrics["n_samples"]),
            "model_gain_median": float(np.nanmedian(gain_eval)) if gain_eval.size else np.nan,
            "model_gain_mean": float(np.nanmean(gain_eval)) if gain_eval.size else np.nan,
            "nearest_inline": nearest_il,
            "nearest_xline": nearest_xl,
            "qc_path": str(qc_path),
            "fig_path": str(fig_path),
        }
        print(
            f"OK: corr {constant_metrics['corr']:.3f}->{model_metrics['corr']:.3f}, "
            f"nmae {constant_metrics['nmae']:.3f}->{model_metrics['nmae']:.3f}"
        )
    except Exception as exc:
        metric_row = {"well_name": well_name, "status": "failed", "error": str(exc)}
        print("FAILED:", exc)
        traceback.print_exc(limit=2)
    well_qc_rows.append(metric_row)

well_qc_metrics_df = pd.DataFrame(well_qc_rows)
well_qc_metrics_df.to_csv(well_qc_metrics_file, index=False)
print("\nSaved", well_qc_metrics_file)
display(well_qc_metrics_df)


## 7) 汇总 QC


In [ ]:
ok_df = well_qc_metrics_df.loc[well_qc_metrics_df["status"] == "ok"].copy()
if ok_df.empty:
    raise ValueError("No successful well QC rows to summarize.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
x = np.arange(len(ok_df))
labels = ok_df["well_name"].tolist()

axes[0].bar(x - 0.18, ok_df["constant_corr"], width=0.36, color="tab:red", alpha=0.65, label="constant")
axes[0].bar(x + 0.18, ok_df["model_gain_corr"], width=0.36, color="tab:blue", alpha=0.75, label="model gain")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha="right")
axes[0].set_ylabel("Correlation")
axes[0].set_title("Synthetic correlation")
axes[0].legend(loc="best")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].bar(x - 0.18, ok_df["constant_nmae"], width=0.36, color="tab:red", alpha=0.65, label="constant")
axes[1].bar(x + 0.18, ok_df["model_gain_nmae"], width=0.36, color="tab:blue", alpha=0.75, label="model gain")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha="right")
axes[1].set_ylabel("NMAE")
axes[1].set_title("Synthetic NMAE")
axes[1].legend(loc="best")
axes[1].grid(True, axis="y", alpha=0.25)

axes[2].bar(x, ok_df["model_gain_median"], color="tab:green", alpha=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=45, ha="right")
axes[2].set_ylabel("Median model gain")
axes[2].set_title("Extracted gain")
axes[2].grid(True, axis="y", alpha=0.25)

fig.tight_layout()
summary_fig = figure_dir / "qc_02_well_model_gain_summary.png"
fig.savefig(summary_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", summary_fig)


## 8) 输出清单


In [ ]:
print("Model:")
print(model_output_file)
print("\nWell QC metrics:")
print(well_qc_metrics_file)
print("\nFigures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
print("\nWell QC CSV:")
for path in sorted(well_qc_dir.glob("model_gain_qc_*.csv")):
    print(path)
